# Quant Skills — Lesson 4: Fat Tails, Correlation and Beta

**Goal:** test the assumptions behind every probability you've used so far — and learn how much your
positions really move together.

1. **Fat tails** — how often big crashes *actually* happen vs. what the bell curve says
2. **Correlation** — whether two positions are two bets or one
3. **Beta** — how much a stock moves when the market moves

**How to run it:** **Runtime → Run all**. Lines you can edit are marked `# <- EDIT`.

## Step 1 — Load the tools

In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, exp, pi

def normal_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

print("Tools loaded.")

## Step 2 — Download five years of daily prices
Five years gives enough history to catch several real selloffs.

In [ ]:
TICKERS = ["SPY", "QQQ", "NVDA", "AMZN"]   # <- EDIT: keep SPY first — it's the market benchmark

raw = yf.download(TICKERS, period="5y", interval="1d", auto_adjust=True, progress=False)
close = raw["Close"]
if isinstance(close, pd.Series):
    close = close.to_frame(TICKERS[0])
close = close[TICKERS].dropna()
returns = close.pct_change().dropna()

MARKET = TICKERS[0]
print(f"{len(returns)} trading days, {returns.index[0].date()} to {returns.index[-1].date()}")

## Step 3 — The bell curve vs. reality

The bars are what the market actually did. The smooth line is the bell curve that Black-Scholes assumes.
Look at the edges: real returns have **more very calm days** and **more extreme days** than the curve.
That's what "fat tails" means.

In [ ]:
r = returns[MARKET]
mu, sd = r.mean(), r.std()

plt.figure(figsize=(10, 4.5))
plt.hist(r * 100, bins=120, density=True, alpha=0.6, label=f"{MARKET} actual daily returns")
x = np.linspace(r.min(), r.max(), 400)
plt.plot(x * 100, np.exp(-((x - mu) / sd) ** 2 / 2) / (sd * sqrt(2 * pi)) / 100,
         linewidth=2, label="Bell curve (what the formula assumes)")
plt.title(f"{MARKET}: real daily returns vs. the bell curve")
plt.xlabel("Daily return (%)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

z = (r - mu) / sd
skew = (z ** 3).mean()
excess_kurt = (z ** 4).mean() - 3
print(f"Skew: {skew:+.2f}   (negative = crashes are bigger than rallies)")
print(f"Excess kurtosis: {excess_kurt:+.2f}   (0 = bell curve; higher = fatter tails)")

## Step 4 — Counting the crashes

How many days moved more than 3, 4 or 5 standard deviations — compared to what a bell curve predicts?

In [ ]:
rows = []
for t in TICKERS:
    rt = returns[t]
    zt = (rt - rt.mean()) / rt.std()
    n = len(zt)
    for k in (3, 4, 5):
        actual = int((zt.abs() > k).sum())
        expected = n * 2 * (1 - normal_cdf(k))
        rows.append({"Ticker": t, "Move bigger than": f"{k} std devs",
                     "Actually happened": actual,
                     "Bell curve predicts": round(expected, 2),
                     "Times more often": "never predicted" if expected < 0.005 else f"{actual/expected:.0f}x"})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"{MARKET}'s five worst days:")
worst = returns[MARKET].nsmallest(5)
for d, v in worst.items():
    print(f"   {d.date()}   {v*100:+.2f}%")

## Step 5 — What this means for your "chance in the money"

In Lesson 3 the formula gave you a probability, like *"14% chance this put finishes in the money."*
That number assumes the bell curve. Here's how often **monthly** drops of different sizes really happened,
compared to what the bell curve says.

In [ ]:
HORIZON = 21   # <- EDIT: trading days (21 ≈ 1 month, 5 ≈ 1 week)

for t in TICKERS:
    m = close[t].pct_change(HORIZON).dropna()
    s = m.std()
    print(f"{t}: typical {HORIZON}-day move ±{s*100:.1f}%")
    for k in (1, 1.5, 2, 2.5):
        actual = (m < m.mean() - k * s).mean()
        model = normal_cdf(-k)
        print(f"   drop worse than {k} std devs ({-k*s*100:.1f}%):  "
              f"bell curve {model:.1%}   actual {actual:.1%}")
    print()
print("If 'actual' beats 'bell curve' for the big drops, the formula is understating your risk on deep puts.")

## Step 6 — Correlation: are your positions really different bets?

Correlation runs from **−1** (always opposite) through **0** (unrelated) to **+1** (always together).
Anything above about **0.8** means the two positions are mostly the same bet.

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(TICKERS))); ax.set_xticklabels(TICKERS)
ax.set_yticks(range(len(TICKERS))); ax.set_yticklabels(TICKERS)
for i in range(len(TICKERS)):
    for j in range(len(TICKERS)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center")
plt.colorbar(im); plt.title("Correlation of daily returns"); plt.show()

big_down = returns[returns[MARKET] < -0.015]
print(f"Correlation on all days vs. on {MARKET}'s big down days (worse than -1.5%):")
for t in TICKERS[1:]:
    print(f"   {MARKET} & {t}:  all days {returns[MARKET].corr(returns[t]):.2f}   "
          f"big down days {big_down[MARKET].corr(big_down[t]):.2f}   ({len(big_down)} days)")
print()
print("Note: on big down days nearly everything falls together — even when correlation looks lower,")
print("check the average move below. That's what hits your account.")
for t in TICKERS[1:]:
    print(f"   On {MARKET}'s big down days, {t} averaged {big_down[t].mean()*100:+.2f}%  "
          f"(vs {MARKET} {big_down[MARKET].mean()*100:+.2f}%)")

## Step 7 — Beta: how much a stock moves with the market

**Beta** answers: *if the market moves 1%, how much does this stock usually move?*
A beta of 2 means roughly twice the market's move. **R²** tells you how much of the stock's movement
the market explains (1.0 = all of it).

In [ ]:
mkt = returns[MARKET]
betas = {}
for t in TICKERS[1:]:
    beta = returns[t].cov(mkt) / mkt.var()
    r2 = returns[t].corr(mkt) ** 2
    betas[t] = beta
    print(f"{t}: beta {beta:.2f}   R² {r2:.2f}   →  if {MARKET} drops 5%, {t} typically drops about {beta*5:.1f}%")

pick = TICKERS[-1]
plt.figure(figsize=(7, 5))
plt.scatter(mkt * 100, returns[pick] * 100, s=6, alpha=0.4)
xs = np.linspace(mkt.min(), mkt.max(), 50)
plt.plot(xs * 100, betas[pick] * xs * 100, color="red", linewidth=2, label=f"beta = {betas[pick]:.2f}")
plt.xlabel(f"{MARKET} daily return (%)"); plt.ylabel(f"{pick} daily return (%)")
plt.title(f"{pick} vs. {MARKET}"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## Step 8 — Your real exposure, in market terms

Type in your positions in dollars. This converts them into **SPY-equivalent dollars** using beta —
the same idea as thinkorswim's beta-weighting, which tells you how much market risk you're really carrying.

In [ ]:
POSITIONS = {"AMZN": 9192, "NVDA": 6494}   # <- EDIT: ticker: dollar value of your position

total_beta_dollars = 0
for t, dollars in POSITIONS.items():
    if t == MARKET:
        b = 1.0
    elif t in betas:
        b = betas[t]
    else:
        print(f"{t}: add it to TICKERS in Step 2 and run all again to get its beta.")
        continue
    total_beta_dollars += dollars * b
    print(f"{t}: ${dollars:,.0f} × beta {b:.2f} = ${dollars*b:,.0f} of {MARKET}-equivalent exposure")

print()
print(f"Total: ${sum(POSITIONS.values()):,.0f} in positions behaves like ${total_beta_dollars:,.0f} of {MARKET}.")
print(f"A 5% {MARKET} drop would typically cost about ${total_beta_dollars*0.05:,.0f}.")

## Your exercises

1. **Step 4:** which ticker has the fattest tails — the most "times more often"?
2. **Step 5:** for NVDA, compare the bell curve and actual numbers for drops worse than 2 std devs.
   What does that mean for the "chance in the money" on a deep NVDA cash-secured put?
3. **Step 6:** what's the correlation between SPY and QQQ? Are puts on both really two separate trades?
4. **Step 8:** add a cash-secured put you're considering. For a put, use **strike × 100 × delta** as the dollar value.

**Next lesson:** backtesting — testing your own fib and ADX setup on years of real history, and the
traps that make backtests lie.